# Konu 12 — Ajan: kararı modele bırakmak

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 10'daki hatta adımların sırasını sen yazdın: önce slogan, sonra görsel. Bugün sırayı **model** seçecek. Modelin
elinde üç araç olacak: görsel çizmek, fotoğraf düzenlemek ve Konu 11'deki gibi Sayfa Kahve'nin marka kılavuzunda aramak.
Bir isteğe hangi araçların, hangi sırayla gerektiğine model karar verecek; araçları çalıştıran yine bizim programımız.

Önce bu işi **elle**, hiçbir çerçeve kullanmadan bir kez yapacağız (Adım 2–4). Sonra aynı işi **Agno** adlı bir ajan
çerçevesiyle yapacağız ve çerçevenin bizim yerimize neler yaptığını göreceğiz.

Isınma anahtarsız. Adım 1'den sonrası `anahtar.txt` ister (Konu 2'deki dosya). Defterin tamamı metin ve arama
modellerine yaklaşık 30 istek gönderir ve 8 kadar görsel üretir ya da düzenler; günlük ücretsiz kotanın %10'undan az.

## Isınma — Ajanı önce sen oyna

Bugün modelin elinde üç araç olacak. Model araçları yalnızca **adlarından** ve **kısa tariflerinden** tanıyacak.
Senin elinde de şimdilik yalnızca bu var:

In [ ]:
tarifler = {
    "afis_ciz": "Sıfırdan yeni bir görsel çizer: afiş, menü kapağı, ilan.",
    "fotograf_duzenle": "Var olan bir fotoğraf dosyasını yönergeye göre değiştirir.",
    "kilavuzda_ara": "Sayfa Kahve'nin marka kılavuzunda arar: renkler, yazı tipleri, ses tonu, yasak sözcükler.",
}

for ad in tarifler:
    print(ad, ":", tarifler[ad])

Aşağıda beş istek var. Model karar vermeden önce **sen** karar ver: her istek için hangi araçlar, hangi sırayla?
`planim` listesinin her elemanı bir liste: içinde araç adları, çalışacakları sırayla. Boş liste (`[]`) "hiçbir araç"
demek. Listede hocanın planı duruyor; katılmadığın yeri değiştir.

In [ ]:
istekler = [
    "kafeye bir afiş yap",
    "veri/fincan.jpg fotoğrafının arka planını sadeleştir",
    "Sayfa Kahve hafta sonu kaçta açılıyor?",
    "Sayfa Kahve için kılavuza uygun bir afiş yap",
    "afişi Instagram'da paylaş",
]
planim = [
    ["afis_ciz"],
    ["fotograf_duzenle"],
    ["kilavuzda_ara"],
    ["kilavuzda_ara", "afis_ciz"],
    [],
]

sira = 0
for istek in istekler:
    print(istek, "->", planim[sira])
    sira = sira + 1

İki listeyi aynı anda gezmek için bir **sıra sayacı** tutuyoruz: `sira` 0'dan başlar, her turda bir artar;
`planim[sira]` o sıradaki istek için planın.

**Sohbete yaz:** dördüncü istekte sıra neden önemli? Beşinci isteği hiçbir araç karşılamıyor: model ne yapmalı?

Adım 6 ve 7'de modelin planını seninkiyle karşılaştıracağız.

## Adım 1 — Hazır parçalar

`ajan_araclari.py` bu klasörde duran yardımcı dosya. Araçların **içinde** çalışacak hazır fonksiyonlar orada:

- `ciz(yonerge, tohum, yol)`: Konu 8'deki fonksiyon. Görseli verilen yola kaydeder; başarılıysa yolu, değilse `None` verir.
- `duzenle(girdi_yolu, yonerge, tohum, yol)`: var olan bir görseli yönergeye göre değiştirir (Konu 7'de arayüzde
  yaptığın iş). `ciz` gibi kaydeder, yolu ya da `None` verir.
- `en_benzer_paragraflar(soru, kac)`: Konu 11'de adım adım kurduğun arama. Kılavuzun paragraflarını vektöre çevirir,
  sorunun vektörüyle karşılaştırır, en benzer `kac` paragrafı tek bir metin olarak verir.
- `modele_sor(soru, sicaklik)`: Konu 5'te yazdığın fonksiyon.

Anahtarı `../anahtar.txt`'den yardımcı dosya kendisi okur; `hesap`, `anahtar`, `basliklar` ve `adres` de oradan geliyor.

In [ ]:
import requests
import matplotlib.pyplot as plt
from ajan_araclari import hesap, anahtar, basliklar, adres
from ajan_araclari import ciz, duzenle, en_benzer_paragraflar, modele_sor, tarifi_goster

Bugün çok görsel açacağız; dört satırı bir fonksiyona koyuyoruz (Konu 8'deki görsel açma satırları). Önce elimizdeki
fotoğrafa bakalım: `veri/fincan.jpg`.

In [ ]:
def goster(yol):
    gorsel = plt.imread(yol)
    plt.imshow(gorsel)
    plt.axis("off")
    plt.show()

goster("veri/fincan.jpg")

Kılavuz araması da hazır. Konu 11'deki gibi bir soru soralım; `2`: en benzer iki paragraf. İlk çalıştırmada kılavuzun
14 paragrafı vektöre çevrilir ve `ciktilar/kilavuz_vektorleri.json` dosyasına saklanır; sonraki aramalar oradan okur.

In [ ]:
bilgi = en_benzer_paragraflar("logonun çevresinde ne kadar boşluk bırakılır?", 2)
print(bilgi)

Gelen paragraflar soruyla ilgili mi? Bu fonksiyon birazdan bir aracın içinde çalışacak: model kılavuza bakmak
isteyince bu metni o alacak.

## Adım 2 — Araç tarifi, elle

Önce çerçevesiz. Modele araçlarımızı **tarif** ediyoruz. Her aracın üç parçası var:

- `name`: aracın adı. Model kararını bu adla bildirecek.
- `description`: aracın tarifi, yani ne yaptığı. Model aracı yalnızca bu cümleden tanır.
- `parameters`: aracın aldığı bilgiler. Burada tek bilgi var, `yonerge`; `"type": "string"` bu bilginin **metin**
  olduğunu, `"required"` zorunlu olduğunu söyler.

Biçim iç içe sözlükler; her katmanı ayrı satırda kuruyoruz. İki aracın adı yardımcı dosyadaki fonksiyonların adıyla
aynı: `ciz` ve `duzenle`. Model fonksiyonları görmez; yalnızca bu tarifleri görür.

In [ ]:
yonerge_alani = {"type": "string", "description": "görselin kısa İngilizce tarifi"}
alanlar = {"yonerge": yonerge_alani}
parametreler = {"type": "object", "properties": alanlar, "required": ["yonerge"]}

ciz_aciklamasi = "Sıfırdan yeni bir görsel çizer: afiş, menü kapağı, ilan."
ciz_tarifi = {"name": "ciz", "description": ciz_aciklamasi, "parameters": parametreler}

duzenle_aciklamasi = "Elimizdeki tek fotoğrafı, veri/fincan.jpg dosyasını, yönergeye göre değiştirir."
duzenle_tarifi = {"name": "duzenle", "description": duzenle_aciklamasi, "parameters": parametreler}

print(duzenle_tarifi)

## Adım 3 — İstekte `tools`, cevapta `tool_calls`

Bu kez soruyu `prompt` ile değil, sohbet biçiminde (`messages`) gönderiyoruz: bir mesaj listesi; her mesajın bir
rolü (`user`: kullanıcı) ve içeriği var. Araç tarifleri gövdenin `tools` alanına gidiyor. Model Konu 2'den beri
kullandığımız llama.

**Çalıştırmadan önce tahmin et, sohbete yaz:** model iki araçtan hangisini seçecek?

In [ ]:
MODEL = "@cf/meta/llama-3.3-70b-instruct-fp8-fast"
metin_adresi = adres(MODEL)

mesaj = {"role": "user", "content": "veri/fincan.jpg fotoğrafının arka planını sadeleştir"}
govde = {"messages": [mesaj], "tools": [ciz_tarifi, duzenle_tarifi]}
cevap = requests.post(metin_adresi, headers=basliklar, json=govde, timeout=60)
print(cevap.status_code)

`200` yazdıysa cevabı açıyoruz. Model bir metin yazmak yerine bir **araç çağrısı** gönderdi: `tool_calls` listesi.
Listedeki her çağrının bir adı (`name`) ve bilgileri (`arguments`) var.

In [ ]:
yanit = cevap.json()
sonuc = yanit["result"]
cagrilar = sonuc["tool_calls"]
cagri = cagrilar[0]
print("Model şunu çağırmak istiyor:", cagri["name"])
print("Bilgiler:", cagri["arguments"])

Model hangi aracı seçti? Tahminin tuttu mu? Yönergeyi hangi dilde yazdı?

Dikkat: model hiçbir şey çizmedi, hiçbir dosyaya dokunmadı. Yalnızca "şu aracı, şu bilgiyle çalıştır" diyen bir
sözlük gönderdi.

## Adım 4 — Aracı çalıştıran program

Kararı uygulayan **bizim programımız**: adı okuyup doğru fonksiyonu çağırıyor. `if / elif / else` GİTA2112'den tanıdık.
`else` dalı bir güvenlik ağı: model listede olmayan bir ad yazarsa program yanlış bir şey çalıştırmaz, durumu söyler.

In [ ]:
ad = cagri["name"]
bilgiler = cagri["arguments"]
yonerge = bilgiler["yonerge"]

if ad == "ciz":
    yol = ciz(yonerge, 42, "ciktilar/elle_cizim.jpg")
elif ad == "duzenle":
    yol = duzenle("veri/fincan.jpg", yonerge, 42, "ciktilar/elle_duzenleme.jpg")
else:
    print("Listede olmayan bir araç:", ad)
    yol = None

print("Sonuç:", yol)

`Sonuç` bir dosya yolu yazdıysa görsele bakalım (`None` yazdıysa bu hücre hata verir; üstteki mesaja bak):

In [ ]:
goster(yol)

Kararı model verdi, işi program yaptı. Ama bir şey eksik: model sonucu hiç görmedi. Bir ajan burada durmaz:

1. Aracın sonucu ("görsel şuraya kaydedildi") yeni bir mesaj olarak modele geri gönderilir.
2. Model sonuca bakıp yeniden karar verir: başka bir araç mı çağıracak, yoksa kullanıcıya cevabını mı yazacak?
3. Yeni bir araç isterse program onu da çalıştırır, sonucu yine geri gönderir...
4. ...ta ki model cevabını yazana ya da bir **sınıra** ulaşılana kadar.

Bu döngüyü elle yazmak mümkün ama uzun: her turda mesaj listesini büyütmek, her aracı adına göre çalıştırmak, sınırı
saymak. **Ajan çerçeveleri** tam olarak bu işi üstlenir. Biz bundan sonra Agno'yu kullanacağız.

## Adım 5 — Agno: model ve araçlar

Agno'da bir aracı tarif etmek için iç içe sözlük kurmuyoruz; **sıradan bir Python fonksiyonu** yazıyoruz. Agno tarifi
fonksiyonun kendisinden çıkarır.

Önce model. Adı yanıltmasın: `openai` paketi ve `OpenAILike` ("OpenAI'ye benzeyen") yalnızca isteğin **biçiminin**
adı. İstek Cloudflare'e gidiyor, model Cloudflare'in ücretsiz modeli; OpenAI hesabı da ödemesi de gerekmiyor.
`base_url` Cloudflare'deki hesabının bu biçimi konuşan adresi.

Ajan için Konu 2'den beri kullandığımız llama yerine `gemma-4` modelini kullanıyoruz. Hocanın denemesinde llama aynı
aracı gereksiz yere birkaç kez çağırdı; gemma her aracı bir kez çağırdı.

In [ ]:
from agno.agent import Agent
from agno.models.openai.like import OpenAILike

taban_adres = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/v1"
model = OpenAILike(id="@cf/google/gemma-4-26b-a4b-it", api_key=anahtar, base_url=taban_adres)

Şimdi üç araç. Her biri yardımcı dosyadaki bir fonksiyonu saran küçük bir fonksiyon. Neden sarıyoruz? Modelin
yalnızca **karar vermesi gereken** bilgileri doldurmasını istiyoruz: yönergeyi model yazsın, tohumu ve dosya yolunu biz
verelim. İlk satırlardaki üç yeni şey:

- `yonerge: str` bir **tür ipucu**: "bu bilgi metin". Adım 2'de elle yazdığın `"type": "string"`'in Python'daki
  karşılığı. `-> str` de "bu fonksiyon metin verir" demek.
- Fonksiyonun ilk satırındaki üç tırnaklı metin bir **docstring**: fonksiyonun tarifi. Agno aracın tarifini
  (`description`) buradan okur. Şimdiye kadarki kodlarda docstring yoktu; burada var, çünkü aracın tarifi tam olarak bu
  metin.
- `return` ettiğimiz metin modele geri gider; model sonraki kararını buna bakarak verir. `print` ise yalnızca bize:
  aracın ne zaman çalıştığını ekranda görelim diye.

In [ ]:
def afis_ciz(yonerge: str) -> str:
    """Sıfırdan yeni bir görsel çizer: afiş, menü kapağı, ilan. Yönerge İngilizce bir görsel tarifi olmalı."""
    print("afis_ciz çalıştı:", yonerge)
    yol = ciz(yonerge, 42, "ciktilar/afis.jpg")
    if yol is None:
        return "Görsel çizilemedi."
    return "Görsel kaydedildi: " + yol

In [ ]:
def fotograf_duzenle(dosya: str, yonerge: str) -> str:
    """Var olan bir fotoğraf dosyasını değiştirir. dosya: fotoğrafın yolu, ör. veri/fincan.jpg. Yönerge İngilizce olmalı."""
    print("fotograf_duzenle çalıştı:", dosya, "|", yonerge)
    yol = duzenle(dosya, yonerge, 42, "ciktilar/duzenlenmis.jpg")
    if yol is None:
        return "Fotoğraf düzenlenemedi."
    return "Düzenlenmiş fotoğraf kaydedildi: " + yol

In [ ]:
def kilavuzda_ara(soru: str) -> str:
    """Sayfa Kahve'nin marka kılavuzunda arar ve soruyla en ilgili üç paragrafı verir.
    Kılavuzda renkler, yazı tipleri, ses tonu, yasak sözcükler, logo ve fotoğraf kuralları, açılış saatleri var."""
    print("kilavuzda_ara çalıştı:", soru)
    return en_benzer_paragraflar(soru, 3)

Agno bu fonksiyonlardan modele ne gönderecek? `tarifi_goster` bunu yazdırır. Adım 2'de elle kurduğun sözlükle karşılaştır:

In [ ]:
tarifi_goster(fotograf_duzenle)

Aynı üç parça: `name` fonksiyonun adından, `description` docstring'den, `parameters` fonksiyonun bilgilerinden ve tür
ipuçlarından. Elle yazdığın iç içe sözlüğü Agno senin yerine kurdu.

## Adım 6 — İlk ajan

Ajanı kuruyoruz: model + araç listesi + iki ayar.

- `tool_call_limit=3`: bir çalıştırmada **en fazla 3 araç** çalışır. Sınır dolunca Agno aracı çalıştırmaz, modele
  "sınır doldu" der. Görsel kotan böyle korunur; her döngünün bir durma koşulu olmalı.
- `telemetry=False`: bunu yazmazsan Agno her çalıştırmada kendi sunucusuna kullanım bilgisi gönderir. Biz göndermiyoruz;
  bugünkü her `Agent`'ta bu ayar var.

Araçları listeye **adlarıyla**, parantezsiz yazıyoruz: fonksiyonu çalıştırmıyoruz, ajana teslim ediyoruz.

In [ ]:
araclar = [afis_ciz, fotograf_duzenle, kilavuzda_ara]
ajan = Agent(model=model, tools=araclar, tool_call_limit=3, telemetry=False)

Isınmadaki ilk istek. `ajan.run` bütün döngüyü çalıştırır; birkaç saniye sürer. `cevap.content` modelin son cevabı.

In [ ]:
cevap = ajan.run("kafeye bir afiş yap")
print(cevap.content)

Son cevap bir cümle; **kararları** göstermiyor. Model hangi araçları, hangi bilgilerle çağırdı? Agno bunları
`cevap.tools` listesinde tutuyor; her çağrının adı `tool_name`, bilgileri `tool_args`. Konu 2'deki `cevap.status_code`
gibi noktayla okunuyorlar. Bu satırları her çalıştırmadan sonra kullanacağız; bir fonksiyona koyalım.

İlk üç satır bir güvenlik ağı: bir sorun çıkarsa (anahtar yanlış, kota doldu, bağlantı koptu) Agno hata vermez;
kırmızı bir `ERROR` satırı yazar, hatayı `cevap.content`'e koyar ve araç listesi hiç oluşmaz (`None`).

In [ ]:
def kararlari_goster(cevap):
    if cevap.tools is None:
        print("Araç kaydı yok. Ajanın son mesajı:", cevap.content)
        return
    sayi = len(cevap.tools)
    print("Çağrılan araç sayısı:", sayi)
    for cagri in cevap.tools:
        print("-", cagri.tool_name, cagri.tool_args)

kararlari_goster(cevap)

Modelin `afis_ciz`'e gönderdiği yönergeye bak. Sen dört sözcük yazdın: model ne yazdı? Hangi dilde? Senin söylemediğin
neleri eklemiş? Kararı modele bırakınca **yönergeyi de model yazıyor**; ne yazdığını okumak senin işin.

In [ ]:
goster("ciktilar/afis.jpg")

Isınmadaki ikinci istek. Bu kez bir dosya yolu da geçiyor:

In [ ]:
cevap = ajan.run("veri/fincan.jpg fotoğrafının arka planını sadeleştir")
kararlari_goster(cevap)
goster("ciktilar/duzenlenmis.jpg")

Model doğru aracı seçti mi? `dosya` bilgisine ne yazdı? Adım 4'te dosya yolunu biz yazmıştık; şimdi o da modelin kararı.

## Adım 7 — Çok adım: sırayı model seçiyor

Isınmadaki dördüncü istek iki araç istiyor: önce kılavuz, sonra çizim. Bu sırayı kimse yazmadı.

**Çalıştırmadan önce tahmin et, sohbete yaz:** model kaç araç çağıracak, hangi sırayla? Isınmadaki planınla aynı mı olacak?

In [ ]:
cevap = ajan.run("Sayfa Kahve için kılavuza uygun bir afiş yap")
kararlari_goster(cevap)

Sıra planınla aynı mı? Model `kilavuzda_ara`'ya ne sordu? `afis_ciz`'e giden yönergede kılavuzdan gelen bir şey var mı:
renkler, insan yüzü olmaması, baskı tarzı? (Model `afis_ciz`'i çağırmadıysa aşağıda bir önceki afişi görürsün.)

In [ ]:
goster("ciktilar/afis.jpg")

Perdenin arkasına bakalım. Agno Adım 4'ün sonundaki döngüyü bizim yerimize çalıştırdı; bütün konuşma `cevap.messages`
listesinde. Her mesajın bir rolü (`role`) ve içeriği (`content`) var:

- `user`: senin isteğin; `assistant`: modelin cevabı; `tool`: aracın modele geri giden sonucu.
- Araç isteyen `assistant` mesajının metni yoktur (`None`): o mesajın içinde yalnızca araç çağrısı vardır.

In [ ]:
for mesaj in cevap.messages:
    print(mesaj.role, "|", mesaj.content)
    print()

Adım 3'te modele tek bir mesaj göndermiştik. Burada her araçtan sonra konuşma büyüyor ve **her `assistant` mesajı
modele giden ayrı bir istek**. Model kılavuza baktıysa paragraflar `tool` mesajında: afişin yönergesini yazarken
onları okudu. Bu, Konu 11'deki "yalnızca şu bilgilere dayanarak" fikrinin ajan hâli; tek fark, bilgiyi ne zaman getireceğine
modelin kendisinin karar vermesi.

## Adım 8 — Tarifi boz, düzelt

Model araçları yalnızca tariflerinden tanıyor. O zaman tarif kötüyse karar da kötü olur. Deneyelim: `kilavuzda_ara`'nın
tarifini bilerek tek, tembel bir cümleye indiriyoruz. Fonksiyonun içi aynı; yalnızca docstring değişti.

Fonksiyonu yeniden tanımladıktan sonra araç listesini ve ajanı da **yeniden kuruyoruz**: eski liste ve eski ajan
eski fonksiyonu tutuyor.

In [ ]:
def kilavuzda_ara(soru: str) -> str:
    """Arama yapar."""
    print("kilavuzda_ara çalıştı:", soru)
    return en_benzer_paragraflar(soru, 3)

araclar = [afis_ciz, fotograf_duzenle, kilavuzda_ara]
ajan = Agent(model=model, tools=araclar, tool_call_limit=3, telemetry=False)

Bu kez istekte "kılavuz" sözcüğü yok. Bir tasarımcı yine de önce kılavuza bakardı: Sayfa Kahve için çalışıyor.

**Çalıştırmadan önce tahmin et:** model kılavuza bakacak mı?

In [ ]:
cevap = ajan.run("Sayfa Kahve için bir menü kapağı çiz")
kararlari_goster(cevap)
goster("ciktilar/afis.jpg")

Model kılavuza baktı mı? Kapağın renkleri kılavuzdaki mürekkep mavisi ve kâğıt beyazı mı? Kılavuzun yasakladığı
bir şey (kırmızı, turuncu, insan yüzü) var mı?

Şimdi tarifi düzeltiyoruz. İyi bir tarif iki şey söyler: araç **ne yapar** (içinde ne var) ve **ne zaman** seçilmeli.

In [ ]:
def kilavuzda_ara(soru: str) -> str:
    """Sayfa Kahve'nin marka kılavuzunda arar ve soruyla en ilgili üç paragrafı verir.
    Kılavuzda renkler, yazı tipleri, ses tonu, yasak sözcükler, logo ve fotoğraf kuralları, açılış saatleri var.
    Sayfa Kahve için görsel ya da metin üretmeden önce kuralları öğrenmek için mutlaka bunu kullan."""
    print("kilavuzda_ara çalıştı:", soru)
    return en_benzer_paragraflar(soru, 3)

araclar = [afis_ciz, fotograf_duzenle, kilavuzda_ara]
ajan = Agent(model=model, tools=araclar, tool_call_limit=3, telemetry=False)

In [ ]:
cevap = ajan.run("Sayfa Kahve için bir menü kapağı çiz")
kararlari_goster(cevap)
goster("ciktilar/afis.jpg")

Karar değişti mi? Kapak değişti mi? Aynı model, aynı istek, aynı fonksiyon; değişen yalnızca bir cümlelik tarif.

Model yanlış aracı seçtiğinde ya da gereken bir aracı atladığında ilk bakılacak yer çoğu zaman model değil, **tariftir**.
Muğlak bir brif, ister insana ister modele verilsin, yanlış işi getirir.

**Sohbete yaz:** bozuk tarifle de model kılavuza baktıysa ne düşünürsün? Tek bir denemeden ne kadar emin olabiliriz?

## Adım 9 — Kendi aracın: `slogan_yaz`

Sıra sende. Ajana dördüncü bir araç veriyoruz: bir konu alıp Türkçe bir slogan yazan `slogan_yaz`. Fonksiyonun içi
hazır (Konu 5'teki `modele_sor`); **docstring'i sen yazacaksın**. Tırnakların arasındaki metni sil ve kendi tarifini yaz:

- Araç ne yapar? Ne verir?
- Ne zaman seçilmeli? (Ör. afiş için metin gerektiğinde.)
- `konu` bilgisine ne yazılmalı? İpucu: model kılavuzdan öğrendiği kuralları (ses tonu, yasak sözcükler) `konu`'ya
  ekleyebilir; tarifinde bunu iste.

İstersen `soru` cümlesini de değiştir; sloganın biçimini o belirliyor.

In [ ]:
def slogan_yaz(konu: str) -> str:
    """BURAYA TARİFİ SEN YAZ."""
    print("slogan_yaz çalıştı:", konu)
    soru = "Şu konu için Türkçe, en fazla altı sözcüklük tek bir slogan yaz. Yalnızca sloganı yaz. Konu: " + konu
    slogan = modele_sor(soru, 0.7)
    if slogan is None:
        return "Slogan yazılamadı."
    return slogan

`tarifi_goster` ile modelin göreceği tarife bak; yazdığın cümle `description`'da mı?

In [ ]:
tarifi_goster(slogan_yaz)

Şimdi dört araçlı ajan. Bu iş üç adım istiyor (kılavuz, slogan, afiş); sınırı 4 yapıyoruz. Sınır işin büyüklüğüne
göre seçilir: beklenen araç sayısı artı biraz pay.

In [ ]:
araclar = [afis_ciz, fotograf_duzenle, kilavuzda_ara, slogan_yaz]
ajan = Agent(model=model, tools=araclar, tool_call_limit=4, telemetry=False)

istek = "Sayfa Kahve'nin yeni sessiz çalışma odası için kılavuza uygun bir slogan yaz, sonra o sloganla bir afiş çiz"
cevap = ajan.run(istek)
kararlari_goster(cevap)
print(cevap.content)

In [ ]:
goster("ciktilar/afis.jpg")

Model senin aracını çağırdı mı? Slogan kılavuzun ses tonuna uyuyor mu (ünlem yok, "indirim", "hemen" yok)? Slogan
afişin yönergesine taşınmış mı, afişte yazı doğru çıkmış mı? Çağırmadıysa tarifini değiştir, iki hücreyi yeniden çalıştır.

Bu afiş bugünün çıktısı: kılavuza bakan, sloganını kendi yazan, sırasını kendi seçen bir programın işi.

## Adım 10 — Bir çalıştırma kaç istek?

Sen bir cümle yazdın; modele kaç istek gitti? Adım 7'de gördük: her `assistant` mesajı ayrı bir istek. Sayalım
(Konu 1'deki sayaç):

In [ ]:
istek_sayisi = 0
for mesaj in cevap.messages:
    if mesaj.role == "assistant":
        istek_sayisi = istek_sayisi + 1

print("Modele giden istek:", istek_sayisi)

Her istekte o ana kadarki bütün konuşma ve bütün araç tarifleri yeniden gidiyor; konuşma büyüdükçe istek de büyüyor.
Agno bu çalıştırmadaki toplam **belirteç** (token) sayısını `cevap.metrics` içinde tutuyor: girdi (modele giden) ve
çıktı (modelin yazdığı). Cloudflare'in fiyat sayfasına göre bu modelde bir milyon girdi belirteci 9.091, bir milyon
çıktı belirteci 27.273 neuron; yani belirteç başına 0,009091 ve 0,027273 (kodda ondalık ayırıcı nokta: `0.009091`).

In [ ]:
olcum = cevap.metrics
girdi = olcum.input_tokens
cikti = olcum.output_tokens
noron = girdi * 0.009091 + cikti * 0.027273
yuvarlak = round(noron)

print("Girdi belirteci:", girdi, "| çıktı belirteci:", cikti)
print("Model istekleri için yaklaşık neuron:", yuvarlak)

Bunun üstüne her `afis_ciz` bir görsel demek: görsel başına yaklaşık 58 neuron (Konu 8'deki hesap). Günlük ücretsiz
kota 10.000 neuron.

**Sohbete yaz:** bu çalıştırmayı günde kaç kez yapabilirdin? Kotayı en çok ne yiyor: modelin kararları mı, görseller mi?
`tool_call_limit` olmasaydı ve model her turda "bir düzenleme daha" deseydi ne olurdu?

## Bonus — Ajanın genel brifi: `instructions`

Docstring tek bir aracı tarif eder. Bütün işi tarif etmek için ajana bir genel yönerge de verilebilir: `instructions`.
Agno bunu konuşmanın başına bir **sistem mesajı** (`system`) olarak koyar; model her kararında onu da okur.

In [ ]:
genel_brif = "Sayfa Kahve için çalışan bir tasarım asistanısın."
genel_brif = genel_brif + " Bir şey üretmeden önce kılavuza bak. Görsel yönergelerini İngilizce yaz."

araclar = [afis_ciz, fotograf_duzenle, kilavuzda_ara]
ajan = Agent(model=model, tools=araclar, instructions=genel_brif, tool_call_limit=3, telemetry=False)
cevap = ajan.run("bir indirim afişi çiz")
kararlari_goster(cevap)

mesajlar = cevap.messages
ilk_mesaj = mesajlar[0]
print(ilk_mesaj.role, "|", ilk_mesaj.content)

İstekte bilerek "indirim" var; kılavuz bu sözcüğü yasaklıyor. Model kılavuza baktı mı? Kullanıcıya itiraz etti mi,
yoksa yine de çizdi mi? Kural bir aracın tarifinde mi daha iyi durur, ajanın genel brifinde mi?